# 🏦 CBDC Transaction Benchmark: PoA vs QBFT
> **Automated framework to evaluate CBDC performance on Hyperledger Besu**
>
> Measures: TPS · Latency · Block Time · Finality
>
> Run each cell top-to-bottom. Full run takes ~15-20 mins.

## ⚙️ Cell 1 — System Setup (Java, Git, Deps)

In [ ]:
# ══════════════════════════════════════════════════════
# CELL 1: System Setup
# ══════════════════════════════════════════════════════

import subprocess, sys, os

def run(cmd, **kw):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    if result.returncode != 0:
        print(result.stderr[-2000:])
    return result

print('Installing Java 21...')
run('apt-get update -qq && apt-get install -y openjdk-21-jre-headless -qq')
print('Java:', run('java -version').stderr.strip()[:60])

print('Installing Python packages...')
run('pip install -q web3 py-solc-x pandas openpyxl matplotlib numpy')

print('Downloading Besu 24.1.2...')
run('mkdir -p /opt/besu')
r = run('wget -q https://hyperledger.jfrog.io/artifactory/besu-binaries/besu/24.1.2/besu-24.1.2.tar.gz -O /tmp/besu.tar.gz')
run('tar -xzf /tmp/besu.tar.gz -C /opt/besu --strip-components=1')
print('Besu:', run('/opt/besu/bin/besu --version').stdout.strip()[:80])

print('\n✅ Setup complete!')


## 📦 Cell 2 — Clone Repository & Upload Dataset

In [ ]:
# ══════════════════════════════════════════════════════
# CELL 2: Git Clone + Dataset Upload
# ══════════════════════════════════════════════════════

import os
from google.colab import files

# ── Option A: Clone from GitHub ───────────────────────
GITHUB_REPO = 'your-username/cbdc-besu-benchmark'  # ← CHANGE THIS
GITHUB_TOKEN = ''  # ← Set for private repos (or use Colab secrets)

# Try to read token from Colab secrets
try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN') or GITHUB_TOKEN
except Exception:
    pass

if GITHUB_TOKEN:
    clone_url = f'https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git'
else:
    clone_url = f'https://github.com/{GITHUB_REPO}.git'

os.chdir('/content')

if os.path.exists('/content/cbdc-besu-benchmark'):
    print('Repo already cloned, pulling latest...')
    os.chdir('/content/cbdc-besu-benchmark')
    run('git pull')
else:
    print(f'Cloning {GITHUB_REPO}...')
    r = run(f'git clone {clone_url} /content/cbdc-besu-benchmark')
    os.chdir('/content/cbdc-besu-benchmark')

print('Working dir:', os.getcwd())

# ── Upload dataset ─────────────────────────────────────
DATASET_PATH = '/content/cbdc-besu-benchmark/FinalDataset.xlsx'

if not os.path.exists(DATASET_PATH):
    print('\n📂 Upload FinalDataset.xlsx:')
    uploaded = files.upload()
    for fname in uploaded:
        os.rename(f'/content/{fname}', DATASET_PATH)
        print(f'✅ Dataset saved: {DATASET_PATH}')
else:
    print(f'✅ Dataset found: {DATASET_PATH}')

# Set env for push script
os.environ['GITHUB_TOKEN'] = GITHUB_TOKEN
os.environ['GITHUB_REPO']  = GITHUB_REPO


## 🚀 Cell 3 — One-Command Benchmark Run

In [ ]:
# ══════════════════════════════════════════════════════
# CELL 3: Run Full Benchmark (One Command)
# ══════════════════════════════════════════════════════

import subprocess, os

os.chdir('/content/cbdc-besu-benchmark')
os.environ['PATH'] = '/opt/besu/bin:' + os.environ['PATH']

# ── CONFIG ─────────────────────────────────────────────
TXNS = 500    # Number of transactions per consensus
TPS  = 50     # Max TPS controller cap
PUSH = False  # Set True to push results to GitHub
# ───────────────────────────────────────────────────────

push_flag = '--push' if PUSH else ''
cmd = f'bash run_benchmark.sh --txns {TXNS} --tps {TPS} --dataset FinalDataset.xlsx {push_flag}'
print(f'Running: {cmd}\n')

process = subprocess.Popen(
    cmd, shell=True, stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT, text=True, bufsize=1
)
for line in process.stdout:
    print(line, end='')
process.wait()

if process.returncode == 0:
    print('\n✅ Benchmark finished successfully!')
else:
    print(f'\n❌ Benchmark failed (exit code {process.returncode})')


## 📊 Cell 4 — Display Results & Graphs

In [ ]:
# ══════════════════════════════════════════════════════
# CELL 4: Display Results
# ══════════════════════════════════════════════════════

import json, pandas as pd
from IPython.display import Image, display
from pathlib import Path

os.chdir('/content/cbdc-besu-benchmark')

# ── Summary table ──────────────────────────────────────
summary_path = Path('results/comparison_summary.csv')
if summary_path.exists():
    df = pd.read_csv(summary_path)
    print('\n=== BENCHMARK SUMMARY ===')
    display(df)

# ── Individual metrics ─────────────────────────────────
for net in ['poa', 'qbft']:
    p = Path(f'results/metrics_{net}.json')
    if p.exists():
        m = json.loads(p.read_text())
        print(f'\n--- {net.upper()} Metrics ---')
        for k, v in m.items():
            print(f'  {k:30s}: {v}')

# ── Graphs ─────────────────────────────────────────────
graphs_dir = Path('results/graphs')
for img in ['dashboard.png', 'tps_comparison.png', 'latency_cdf.png', 'block_times.png']:
    img_path = graphs_dir / img
    if img_path.exists():
        print(f'\n--- {img} ---')
        display(Image(str(img_path), width=850))


## 💾 Cell 5 — Download Results

In [ ]:
# ══════════════════════════════════════════════════════
# CELL 5: Download All Results as ZIP
# ══════════════════════════════════════════════════════

import shutil, os
from google.colab import files

os.chdir('/content/cbdc-besu-benchmark')

# Package results
shutil.make_archive('/content/cbdc_benchmark_results', 'zip', 'results')
files.download('/content/cbdc_benchmark_results.zip')
print('✅ Results downloaded!')


## 🔁 Cell 6 — Push Results to GitHub (Optional)


In [ ]:
# ══════════════════════════════════════════════════════
# CELL 6: Push to GitHub
# ══════════════════════════════════════════════════════
# Set GITHUB_TOKEN in Colab Secrets (🔑 icon in sidebar)

import subprocess, os

try:
    from google.colab import userdata
    os.environ['GITHUB_TOKEN'] = userdata.get('GITHUB_TOKEN') or ''
    os.environ['GITHUB_REPO']  = userdata.get('GITHUB_REPO') or 'your-username/cbdc-besu-benchmark'
except Exception:
    pass

os.chdir('/content/cbdc-besu-benchmark')
result = subprocess.run('bash scripts/push_results.sh', shell=True,
                        capture_output=True, text=True)
print(result.stdout)
if result.stderr: print(result.stderr)
